<a href="https://colab.research.google.com/github/abhilash-790/CodeAlpha-web-Scraping/blob/main/Task1_Web_Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1: Web Scraping

**Goal:** Extract data from a public web page, build a clean custom dataset, and save it as a CSV file.

**Tools used:** `requests` + `BeautifulSoup` (Python libraries)

**Target site:** [quotes.toscrape.com](http://quotes.toscrape.com) — a website built specifically for practicing web scraping, so it's safe and legal to scrape.

**What this notebook does:**
1. Fetches HTML pages from the site
2. Parses the HTML structure with BeautifulSoup
3. Navigates the page to pull out quotes, authors, and tags
4. Handles pagination to collect data across multiple pages
5. Cleans the data and stores it in a pandas DataFrame
6. Exports the final dataset to `quotes_dataset.csv`


## 1. Install & import libraries

Run the cell below once if the packages aren't already installed.

In [1]:
# Uncomment the line below if you need to install the packages
# !pip install requests beautifulsoup4 pandas lxml

import time
import requests
from bs4 import BeautifulSoup
import pandas as pd


## 2. Fetch a single page and inspect the HTML structure

Before scraping everything, it's good practice to look at the structure of one page first.

In [2]:
BASE_URL = "http://quotes.toscrape.com"

response = requests.get(BASE_URL, timeout=10)
print("Status code:", response.status_code)

soup = BeautifulSoup(response.text, "html.parser")

# Peek at the first quote block to understand the HTML structure
first_quote = soup.find("div", class_="quote")
print(first_quote.prettify()[:800])


Status code: 200
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
 <span class="text" itemprop="text">
  “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
 </span>
 <span>
  by
  <small class="author" itemprop="author">
   Albert Einstein
  </small>
  <a href="/author/Albert-Einstein">
   (about)
  </a>
 </span>
 <div class="tags">
  Tags:
  <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="keywords"/>
  <a class="tag" href="/tag/change/page/1/">
   change
  </a>
  <a class="tag" href="/tag/deep-thoughts/page/1/">
   deep-thoughts
  </a>
  <a class="tag" href="/tag/thinking/page/1/">
   thinking
  </a>
  <a class="tag" href="/tag/world/page/1/">
   world
  </a>
 </div>
</div>



## 3. Write a function to parse one page

Each quote is inside a `<div class="quote">` element, containing:
- the quote text in a `<span class="text">`
- the author's name in a `<small class="author">`
- a list of tags in `<a class="tag">` elements


In [3]:
def parse_page(soup):
    """Extract all quotes from a single parsed page."""
    records = []
    quote_blocks = soup.find_all("div", class_="quote")

    for block in quote_blocks:
        text = block.find("span", class_="text").get_text(strip=True)
        author = block.find("small", class_="author").get_text(strip=True)
        tags = [tag.get_text(strip=True) for tag in block.find_all("a", class_="tag")]

        records.append({
            "quote": text,
            "author": author,
            "tags": ", ".join(tags)
        })

    return records


## 4. Handle pagination and collect data from every page

The site has a "Next →" link at the bottom of each page. We follow it until there isn't one.

In [4]:
all_records = []
url = BASE_URL
page_num = 1

while url:
    print(f"Scraping page {page_num}: {url}")
    response = requests.get(url, timeout=10)
    soup = BeautifulSoup(response.text, "html.parser")

    all_records.extend(parse_page(soup))

    # Look for a "Next" button to move to the following page
    next_btn = soup.find("li", class_="next")
    if next_btn:
        next_href = next_btn.find("a")["href"]
        url = BASE_URL + next_href
        page_num += 1
        time.sleep(1)  # be polite to the server
    else:
        url = None

print(f"\nDone. Collected {len(all_records)} quotes across {page_num} pages.")


Scraping page 1: http://quotes.toscrape.com
Scraping page 2: http://quotes.toscrape.com/page/2/
Scraping page 3: http://quotes.toscrape.com/page/3/
Scraping page 4: http://quotes.toscrape.com/page/4/
Scraping page 5: http://quotes.toscrape.com/page/5/
Scraping page 6: http://quotes.toscrape.com/page/6/
Scraping page 7: http://quotes.toscrape.com/page/7/
Scraping page 8: http://quotes.toscrape.com/page/8/
Scraping page 9: http://quotes.toscrape.com/page/9/
Scraping page 10: http://quotes.toscrape.com/page/10/

Done. Collected 100 quotes across 10 pages.


## 5. Load the data into a DataFrame

In [5]:
df = pd.DataFrame(all_records)
df.head(10)


,quote,author,tags
0,“The world as we have created it is a process ...,Albert Einstein,"change, deep-thoughts, thinking, world"
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"abilities, choices"
2,“There are only two ways to live your life. On...,Albert Einstein,"inspirational, life, live, miracle, miracles"
3,"“The person, be it gentleman or lady, who has ...",Jane Austen,"aliteracy, books, classic, humor"
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"be-yourself, inspirational"
5,“Try not to become a man of success. Rather be...,Albert Einstein,"adulthood, success, value"
6,“It is better to be hated for what you are tha...,André Gide,"life, love"
7,"“I have not failed. I've just found 10,000 way...",Thomas A. Edison,"edison, failure, inspirational, paraphrased"
8,“A woman is like a tea bag; you never know how...,Eleanor Roosevelt,misattributed-eleanor-roosevelt
9,"“A day without sunshine is like, you know, nig...",Steve Martin,"humor, obvious, simile"


In [6]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   quote   100 non-null    object
 1   author  100 non-null    object
 2   tags    100 non-null    object
dtypes: object(3)
memory usage: 2.5+ KB


## 6. Basic cleaning & quick exploration

In [7]:
# Drop exact duplicate rows, if any
df = df.drop_duplicates().reset_index(drop=True)

# Top authors by number of quotes
df["author"].value_counts().head(10)


,count
author,
Albert Einstein,10
J.K. Rowling,9
Marilyn Monroe,7
Dr. Seuss,6
Mark Twain,6
Jane Austen,5
C.S. Lewis,5
Bob Marley,3
Mother Teresa,2


In [8]:
# Most common tags
tag_series = df["tags"].str.split(", ").explode()
tag_series.value_counts().head(10)


,count
tags,
love,14
inspirational,13
life,13
humor,12
books,11
reading,7
friendship,5
friends,4
truth,4


## 7. Save the custom dataset to CSV

In [9]:
output_path = "quotes_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} rows to {output_path}")


Saved 100 rows to quotes_dataset.csv


## Notes / next steps

- This notebook targets `quotes.toscrape.com` because it's explicitly meant for scraping practice. When scraping real sites, always check their `robots.txt` and Terms of Service first.
- To scrape a **different** site, you mainly need to change: the `BASE_URL`, the tag/class names inside `parse_page`, and the pagination logic in the loop.
- For JavaScript-heavy sites (where data loads dynamically), `requests` + `BeautifulSoup` won't see the content — you'd need `Selenium` or `Scrapy` with a headless browser instead.
- No-code alternatives mentioned in the task (Octoparse, ParseHub) do the same job through a visual interface instead of code.
